# AI Call Moderator v5 — Real-Time Assembly Line + LIVE GUI Dashboard\n\nSame pipeline as v4, now streaming every judged turn to a **live web dashboard**\n(started automatically by `run_vllm_server.sh` on port 7860): rep and customer\nhighlighted on opposite sides with their identities, violations highlighted red with\npolicy-code chips, and a flashing escalation banner the instant a rule trips.

Implements the architecture document's **3-step assembly line** for the ROCm Jupyter lab,
moderating 1–3 real Kaggle recordings with near-instant flagging. **Speed is the metric**:
every hop is timed, and the headline number is *transcript-ready → flag raised* in milliseconds.

```
STEP 1  THE EARS   audio in 5s chunks ──> Faster-Whisper (CPU int8) ──> transcript
                                  │  zero-copy, in-process
                                  ▼  asyncio.Queue  (one FIFO per call = order preserved)
STEP 2  THE BRAIN  worker awaits queue ──> regex pre-screen (0 tokens) ──> vLLM guided-JSON judge
                                  │  escalation rules (deterministic code)
                                  ▼  alert asyncio.Queue
STEP 3  THE ALARM  consumer flags INSTANTLY (flushed print) + sqlite :memory: audit DB
```

**Architecture-document mapping** (what was kept / changed and why):

| Document said | v4 does | Reason |
|---|---|---|
| Faster-Whisper, 5s chunks | ✅ same, CPU int8 | CTranslate2 has no ROCm backend; CPU int8 beats real time on 5s chunks and leaves 100% of the GPU to vLLM |
| in-memory `asyncio.Queue` handshakes | ✅ same — one queue **per call** | zero-copy, ~0 ms hop; per-call FIFO keeps turn order (escalation rules need it) while calls run in parallel |
| Qwen 2.5 32B Q6 via vLLM | Qwen3-4B-Instruct via vLLM | 8× fewer weights = lower latency & tokens; the judging task doesn't need 32B |
| PyQt6 GUI + pyqtSignal + OS alarm | flushed alert print + sqlite `:memory:` | no desktop GUI inside Jupyter; the handshake & latency measurement are identical |
| WebRTC/WebSocket ingress | chunked file streaming (optional real-time pacing) | same dataflow; swap the producer's source for a socket later — nothing downstream changes |
| (LangChain/LangGraph considered) | **not used** | a fixed 3-stage pipeline needs no graph framework; it would add overhead between transcript and flag |

## STEP 0 — Tab 1: spin up the server (one command)

Open a **Terminal tab** (`+` → Terminal) and run:

```bash
bash run_vllm_server.sh
```

It installs all pip requirements, repairs the starlette/fastapi conflict, and launches vLLM.
Leave it running; wait for `Application startup complete` (first run downloads ~8 GB).
Then come back here — **Tab 2** — and run the cells below one by one.

In [ ]:
# ── Why asyncio + stdlib over LangChain/LangGraph ──────────────────────────────
# The pipeline is a fixed 3-stage DAG (audio → text → verdict → flag) with no
# branching, retry loops, or tool-calling that would justify a graph framework.
# LangChain/LangGraph add ~100 ms+ orchestration overhead per turn on top of
# each LLM call — significant when the goal is sub-second flag latency.
# asyncio.Queue gives zero-copy ~µs-latency handshakes between stages with a
# simpler call graph.  The OpenAI-compatible client lets us swap the local vLLM
# server for any hosted endpoint without touching any pipeline code.
# ============ CELL 1: clients + health check ============
import os, json, re, time, asyncio, sqlite3            # stdlib only: parsing, timing, concurrency, in-memory DB
from collections import defaultdict, deque             # defaultdict: metric buckets / deque: rolling context window
from openai import OpenAI, AsyncOpenAI                 # OpenAI-compatible clients for the LOCAL vLLM server

BASE_URL          = "http://localhost:8000/v1"         # where Tab 1's vLLM listens
SERVED_MODEL_NAME = "call-moderator-llm"               # must match --served-model-name in the runner
API_KEY           = "local-key-123"                    # must match --api-key in the runner

sync_client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)        # used once, for the health check below
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)   # used by the pipeline (async, non-blocking)

try:                                                   # health check: fail with INSTRUCTIONS, not a stack trace
    available_models = [m.id for m in sync_client.models.list().data]
except Exception:
    raise SystemExit("Cannot reach vLLM at " + BASE_URL + ".\n"
                     "Tab 1 first: bash run_vllm_server.sh  ->  wait for 'Application startup complete'.")
print("vLLM is serving:", available_models)            # expect ['call-moderator-llm']
assert SERVED_MODEL_NAME in available_models           # wrong --served-model-name would trip this

## CELL 2 — Policy, escalation rules & zero-token pre-screener

The rules the Brain enforces. Codes (`C1`, `R2`, …) keep prompts and outputs tiny — token
efficiency starts here. The regex pre-screener costs zero tokens and only produces *hints*
for the LLM to verify (a rep **refusing** an SSN must not be auto-flagged).

In [ ]:
# ── Why regex pre-screener + deterministic escalation rules ─────────────────────
# Two-tier detection keeps per-turn latency and token spend minimal:
#   Tier 1 — keyword regex (this cell): ~microseconds, zero tokens.  Produces
#     HINT codes that tell the judge exactly where to look, cutting false-
#     negatives on obvious violations without any LLM round-trip.
#   Tier 2 — LLM judge (CELL 3): verifies hints and catches subtler violations
#     (e.g. a rep REFUSING an SSN request looks like C1 to a keyword scanner;
#     the LLM resolves context correctly).  Escalation stays in pure Python
#     (rule 1/2/3) — every trigger is auditable without a second LLM call.
# ============ CELL 2: policy + pre-screener ============
POLICY = {                                             # code -> (severity, definition)
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes, off-the-books deals, "
                       "falsifying records, or advising someone to lie"),
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50 or outcomes "
                       "outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}
SEVERITY_BY_CODE = {code: severity for code, (severity, _) in POLICY.items()}   # quick lookup for the rules

COMPANY_POLICY_ALLOWANCES = (                          # what reps ARE allowed -> lets the judge tell refusal from violation
    "Refunds up to $50 may be issued instantly; larger amounts require supervisor approval. "
    "Identity verification = full name + last 4 of account + billing ZIP (NEVER full SSN, "
    "full card number, CVV, or password). Reps may offer one goodwill credit up to $25 per call.")

# Escalation = DETERMINISTIC CODE (auditable), not the model:
#   rule 1: any critical violation            -> flag immediately
#   rule 2: two high-severity violations      -> flag
#   rule 3: customer sentiment <= -2 twice in a row -> flag (supervisor assist)

PRESCREEN_PATTERNS = {                                 # obvious keyword tells, compiled per turn at ~microsecond cost
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)", r"\bi (promise|guarantee)\b"],
}

def keyword_prescreen(text: str) -> list:              # returns hint codes; 0 tokens, deterministic
    lowered = text.lower()                             # case-insensitive matching
    return [code for code, patterns in PRESCREEN_PATTERNS.items()
            if any(re.search(pattern, lowered) for pattern in patterns)]

print(f"{len(POLICY)} violation codes loaded; pre-screener ready")

## CELL 3 — Instrumented, bounded, schema-constrained generation

Every LLM call goes through `generate_json`: an **`asyncio.Semaphore`** bounds in-flight
requests (the client-side gate; vLLM's own scheduler queues/batches beyond it), **guided
JSON** makes off-schema output impossible, and every call records exact **tokens** (from
vLLM's `usage`) and **latency** per pipeline stage.

In [ ]:
# ── Why guided JSON + Semaphore(16) + temperature=0 ────────────────────────────
# guided_json: vLLM grammar-constrains the sampler to the schema DURING decoding,
#   making off-schema tokens physically unsamplable.  Eliminates JSON-repair
#   retries and their latency cost entirely vs. post-hoc parse-and-retry loops.
# Semaphore(16): without this gate a bulk run opens hundreds of simultaneous
#   sockets, thrashing the asyncio event loop.  16 keeps the GPU saturated
#   (vLLM batches internally) without exceeding its scheduler budget.
# temperature=0: greedy decoding for reproducible verdicts — the same transcript
#   must always produce the same violation codes to pass a compliance audit.
# ============ CELL 3: generate_json (the only door to the LLM) ============
MAX_CONCURRENT_LLM_REQUESTS = 16                       # client-side cap; protects sockets at scale, GPU stays saturated
llm_request_slot = asyncio.Semaphore(MAX_CONCURRENT_LLM_REQUESTS)   # async gatekeeper shared by all workers

STAGE_TOKEN_USAGE = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0})  # tokens per pipeline stage
STAGE_LATENCY_MS  = defaultdict(list)                  # wall-clock ms per call, per stage (for avg/p95/max later)

async def generate_json(stage: str, system_prompt: str, user_prompt: str,
                        json_schema: dict, max_tokens: int = 64) -> dict:
    async with llm_request_slot:                       # wait here if 16 requests are already in flight
        started = time.perf_counter()                  # latency clock starts at dispatch
        response = await async_client.chat.completions.create(
            model=SERVED_MODEL_NAME,                   # the vLLM-served judge
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_prompt}],
            temperature=0,                             # greedy decoding -> reproducible verdicts
            max_tokens=max_tokens,                     # hard output budget per call
            extra_body={"guided_json": json_schema,    # vLLM grammar-constrains sampling to this schema
                        "chat_template_kwargs": {"enable_thinking": False}},  # Qwen3: belt+braces no-think
        )
    STAGE_LATENCY_MS[stage].append((time.perf_counter() - started) * 1000)   # record ms
    bucket = STAGE_TOKEN_USAGE[stage]                  # accumulate exact server-reported token counts
    bucket["calls"]      += 1
    bucket["prompt"]     += response.usage.prompt_tokens
    bucket["completion"] += response.usage.completion_tokens
    try:
        return json.loads(response.choices[0].message.content)   # guided JSON -> this should never fail
    except json.JSONDecodeError:
        return {}                                      # belt-and-braces: empty verdict instead of a crash

# quick smoke test: proves server reachability, guided decoding, and metering in one shot
smoke = await generate_json("smoke_test", "Rate sentiment.",
                            'Customer said: "this is great, thanks!"',
                            {"type": "object", "properties": {"sentiment": {"enum": [-2, -1, 0, 1, 2]}},
                             "required": ["sentiment"]}, max_tokens=8)
print("guided JSON:", smoke, "| latency:", f"{STAGE_LATENCY_MS['smoke_test'][0]:.0f} ms")

In [ ]:
# ── OPTIONAL: Langfuse monitoring — three modes, NO ACCOUNT required ────────────
#
# MODE A: Self-hosted Docker (recommended — free, no account, full dashboard)
#   docker run -d --name langfuse -p 3000:3000 \
#     -e NEXTAUTH_SECRET=change-me -e SALT=change-me \
#     -e DATABASE_URL=file:/data/langfuse.db -v langfuse_data:/data \
#     langfuse/langfuse:latest
#   → open http://localhost:3000, create project, copy keys to .env
#   → set LANGFUSE_HOST=http://localhost:3000 in .env
#
# MODE B: Langfuse cloud (free tier — sign up at langfuse.com, add keys to .env)
#
# MODE C: Local file (zero setup, zero account — traces saved to langfuse_traces.jsonl)
#   Just run this cell without a .env file. View traces with:
#   langfuse_config.show_local_traces()
#
# See README.md for full setup steps. Pipeline is UNCHANGED if you skip this cell.
# ────────────────────────────────────────────────────────────────────────────────
import importlib, langfuse_config
importlib.reload(langfuse_config)   # re-run load_dotenv() every time so keys
                                    # are picked up even on first cell execution
generate_json = langfuse_config.patch_generate_json(
    generate_json, STAGE_TOKEN_USAGE, SERVED_MODEL_NAME)
# To view local traces (Mode C): langfuse_config.show_local_traces()
# To flush before shutdown:       langfuse_config.flush()

# ── Sessions ──────────────────────────────────────────────────────────────────
# Every pipeline run generates a unique session_id (e.g. sess-1781397200).
# All LLM judge calls in that run are attached to this session as a first-class
# Langfuse field (not just metadata) via a parent span that carries session_id
# and user_id=call_id.
#
# To view a session:
#   Langfuse cloud UI → Sessions (left sidebar) → select session ID
#
# The Sessions view shows aggregate token usage, total cost, and a latency
# timeline across all calls in that run. Each call is also individually
# filterable in the Traces view by user_id=call_id.


## CELL 3b — Connect to the live GUI dashboard

The runner already started `gui_server.py` on **port 7860** next to vLLM. The pipeline
fires every judged turn and every escalation at it via `emit_event` (fire-and-forget;
if the GUI isn't running the pipeline is unaffected). Two ways to view it:

1. **Browser tab** — open `<your Jupyter base URL>/proxy/7860/` (jupyter-server-proxy).
   The cell below prints the exact URL for this lab.
2. **Embedded** — the IFrame below renders the dashboard right inside the notebook.

In [ ]:
# ── Why httpx AsyncClient, fire-and-forget, 2s per-request timeout ─────────────
# httpx is chosen over aiohttp for its cleaner async API and per-request timeout
# support (not just per-connection).  The 2s ceiling means a slow or dead GUI
# server never adds more than 2s to a live call judgment — the pipeline degrades
# gracefully rather than blocking.  GUI_AVAILABLE + 10s retry-after back-off
# means a mid-run GUI restart reconnects automatically without any intervention.
# ============ CELL 3b: GUI event emitter + embedded dashboard ============
import os
import httpx as gui_httpx
from IPython.display import IFrame, display

GUI_PORT      = 7860
GUI_EVENT_URL = f"http://localhost:{GUI_PORT}/event"   # pipeline -> dashboard (same box)
gui_client    = gui_httpx.AsyncClient(timeout=2.0)     # tiny budget: never slow the pipeline
GUI_AVAILABLE = True                                   # backs off when the GUI is down...
gui_retry_after = 0.0                                  # ...but RETRIES every 10s, so a GUI restart
                                                       # mid-run reconnects automatically

async def emit_event(event: dict):
    """Fire-and-forget push to the dashboard. The pipeline NEVER waits on the GUI."""
    global GUI_AVAILABLE, gui_retry_after
    if not GUI_AVAILABLE and time.monotonic() < gui_retry_after:
        return                                         # still in back-off — skip quietly
    try:
        await gui_client.post(GUI_EVENT_URL, json=event)
        GUI_AVAILABLE = True                           # success -> (re)connected
    except Exception:
        GUI_AVAILABLE = False                          # GUI down -> back off 10s, keep moderating
        gui_retry_after = time.monotonic() + 10.0

# Where to open the dashboard in a browser tab (JupyterHub prefixes the proxy path)
jupyter_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
dashboard_path = f"{jupyter_prefix}proxy/{GUI_PORT}/"
print(f"Dashboard URL (open in a new browser tab): {dashboard_path}")
print("(prepend your lab host, e.g. https://<lab-host>" + dashboard_path + ")")

await emit_event({"type": "status", "text": "notebook connected — pipeline ready"})
display(IFrame(src=dashboard_path, width="100%", height=560))   # embedded view

## CELL 4 — Load recordings from the repo (Kaggle retired)

v5 no longer pulls from Kaggle: the recordings are **committed in `v5/call_recordings/`**,
so anyone who clones the branch can run everything with zero accounts and zero downloads.
(A leftover `kaggle_call_data/` from older runs is also accepted.)

*Dataset attribution: "Call Center Audio Dataset" © Unidata
(kaggle.com/datasets/unidpro/call-center-audio), CC BY-NC-ND 4.0 —
redistributed unmodified for non-commercial use.*

In [ ]:
# ── Why committed recordings over runtime download ───────────────────────────
# Committing audio files (WAV/MP3/FLAC) to the repo eliminates external auth,
# token-expiry failures, and network latency from run setup — anyone who clones
# the branch can run the full pipeline with zero accounts and zero downloads.
# AUDIO_DIRS lists every folder that may contain recordings; the pipeline
# searches all of them so call_recordings/ and scam_call/ are treated equally.
# ============ CELL 4: discover recordings across all audio folders ============
import pathlib

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

# ── Add or remove folders here — all are searched at startup ─────────────────
AUDIO_DIRS = [
    pathlib.Path("call_recordings"),   # committed team/demo recordings
    pathlib.Path("scam_call"),         # scam-call samples
]

def collect_audio(directory: pathlib.Path) -> list:
    if not directory.exists():
        return []
    return sorted(p for p in directory.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)

audio_files = []
for _d in AUDIO_DIRS:
    _found = collect_audio(_d)
    audio_files.extend(_found)
    status = f"{len(_found)} file(s)" if _found else "not found or empty"
    print(f"  {_d}: {status}")

transcript_files = []                                   # kept for downstream compatibility

if not audio_files:
    raise RuntimeError(
        "No audio files found — AUDIO_DIRS contains no recordings.\n"
        "Add WAV/MP3/FLAC files to call_recordings/ or scam_call/ then re-run this cell."
    )
print(f"\n{len(audio_files)} recording(s) available:")
for _r in audio_files:
    print("  ", _r)


## CELL 4c — Assess the dataset (audio QA before trusting transcripts)

Published call datasets are usually **redacted**: PII and profanity are overwritten with censor
bleeps, and Whisper notoriously *hallucinates words over pure tones and silence* — which can
produce false flags downstream. This cell profiles every recording before the pipeline runs:
duration, channels, silence share, and how many 5s chunks look like **pure tones** (one FFT
bin dominating the spectrum = beep, not speech). High beep/silence counts mean transcripts
need the confidence gates that CELL 5 now applies (`no_speech_prob`, `avg_logprob`,
`compression_ratio` filters).

In [ ]:
# ── What is RMS (Root Mean Square)? ────────────────────────────────────────────
# RMS = √( mean of all sample² values ) — it measures the effective "power level"
# of an audio waveform.  Why it matters here:
#   • Silence detection: if a chunk's RMS is near zero (< 1e-4 abs mean),
#     there's no signal worth transcribing — skip it before paying ASR cost.
#   • Beep detection: a censor tone has very high, narrow-band energy
#     (one FFT bin dominates), unlike speech which spreads RMS across
#     hundreds of harmonic frequencies.
#   • Volume consistency: preprocessing targets a fixed RMS level so
#     Whisper/Vosk receives stable input regardless of recording equipment.
# The `float(np.abs(piece).mean()) < 1e-4` check below IS an RMS-style gate.
# ── Why FFT-based beep detection + pre-run QA with librosa ─────────────────────
# Published call datasets redact PII with censor bleeps (constant-frequency
# tones).  Whisper hallucinates plausible words over pure tones — running this
# QA cell confirms how many beep/silence chunks exist before the pipeline runs,
# so we know the confidence gates in CELL 5 are load-bearing.
# FFT threshold 0.30 (peak/total energy): a single-frequency tone concentrates
# >60% of energy in one bin; voiced speech spreads energy across hundreds of
# harmonics.  0.30 is a robust midpoint.  librosa.load handles resampling and
# channel layout in one call, avoiding a separate ffmpeg dependency.
# ============ CELL 4c: dataset audio QA ============
import numpy as np, librosa

def profile_recording(audio_path, chunk_seconds=5.0, sample_rate=16000):
    samples, _ = librosa.load(audio_path, sr=sample_rate, mono=False)   # keep channel layout
    channels = samples if samples.ndim == 2 else samples[np.newaxis, :]
    chunk_len = int(chunk_seconds * sample_rate)
    silent_chunks = beep_chunks = speech_chunks = 0
    for channel in channels:                            # walk every 5s window on every channel
        for start in range(0, channel.shape[0], chunk_len):
            piece = channel[start:start + chunk_len]
            if piece.size == 0 or float(np.abs(piece).mean()) < 1e-4:
                silent_chunks += 1                      # effectively no signal
                continue
            windowed = piece * np.hanning(len(piece))   # tone test: one FFT bin dominating = beep
            spectrum = np.abs(np.fft.rfft(windowed))
            if spectrum.sum() > 1e-6 and spectrum.max() / spectrum.sum() > 0.30:
                beep_chunks += 1
            else:
                speech_chunks += 1
    duration_s = channels.shape[1] / sample_rate
    return {"file": audio_path.name, "duration_s": round(duration_s, 1),
            "channels": channels.shape[0], "speech_chunks": speech_chunks,
            "beep_chunks": beep_chunks, "silent_chunks": silent_chunks}

print(f"{'file':<40}{'dur(s)':>8}{'ch':>4}{'speech':>8}{'beeps':>7}{'silent':>8}")
print("-" * 78)
for audio_path in audio_files:
    p = profile_recording(audio_path)
    print(f"{p['file']:<40}{p['duration_s']:>8}{p['channels']:>4}"
          f"{p['speech_chunks']:>8}{p['beep_chunks']:>7}{p['silent_chunks']:>8}")
print("\nbeep_chunks > 0 confirms censor bleeps -> CELL 5's confidence gates protect the judge")

## CELL 5 — STEP 1: THE EARS (Faster-Whisper producer)

The producer streams each recording in **5-second chunks** (the document's chunk size),
transcribes each chunk with **Faster-Whisper `small` int8 on CPU** — deliberate: CTranslate2
has no ROCm backend, CPU int8 still beats real time on 5s chunks, and the GPU stays 100%
vLLM's — then puts the transcript on the call's **`asyncio.Queue`** stamped with
`transcribed_at` (the moment the flag-latency clock starts).

Stereo recordings get each channel transcribed separately (telephony puts one party per
channel — free speaker separation). `REALTIME_PACING=True` makes the producer sleep 5s per
chunk to mimic a live stream; leave `False` to benchmark flat-out.

In [ ]:
print(f"THE EARS ready (faster-whisper {WHISPER_MODEL_SIZE} / int8 / CPU — mode set by Cell 8)")


## CELL 6 — STEP 2: THE BRAIN (queue consumer → vLLM judge [Qwen3-4B-Instruct] → escalation rules)

One Brain worker **per call** pulls from that call's queue with `await queue.get()` —
FIFO order within a call is preserved (rule 3 needs consecutive sentiment), while multiple
calls run in parallel and vLLM batches their requests together.

Speaker role is resolved by a **3-tier pipeline** before every judge call:
1. **Stereo channel metadata** — if the recording is stereo, ch0=rep / ch1=customer is telephony
   ground-truth; no inference needed.
2. **Keyword regex pre-pass** (`_guess_mono_speaker`) for mono — matches obvious lexical markers
   in microseconds at zero token cost. Customer markers: *"your agent"*, *"loyal member"*,
   *"you work for me"*, *"get me your manager"*, *"the customer is always right"*.
   Rep markers: *"how can I help"*, *"let me pull up"*, *"one moment"*, *"you were speaking with"*.
   Returns `'rep'`, `'customer'`, or `None` (ambiguous → tier 3).
3. **LLM `speaker` field** — fallback only when regex is ambiguous. Prompt passes the previous
   turn's role so the judge can use alternation context within the tight `max_tokens` budget.

Micro-utterances (≤12 chars, no regex hints) skip the judge entirely — saving ~350 tokens
and ~300 ms per short turn. The judge prompt holds only the last 3 utterances (rolling window).
Escalation is deterministic Python (rule 1/2/3); on a flag the worker pushes to the **alert queue**
(handshake 2→3) and keeps processing.

In [ ]:
# ── Why one worker per call, 3-turn context window, ack fast-path ──────────────
# One asyncio worker per call: preserves strict FIFO order within each call
# (rule 3 reads CONSECUTIVE customer sentiment — interleaving corrupts it) while
# letting multiple calls run in parallel and letting vLLM batch them.
# 3-turn context: enough for the judge to see the exchange that produced the
# latest utterance without inflating token count.  Every extra turn multiplies
# total token spend linearly across all turns in a run.
# Ack fast-path for micro-utterances (<=12 chars, no regex hints): short turns
# like 'Okay.' represent ~20% of real call turns but carry zero policy signal.
# Skipping the judge saves ~350 tokens and ~300 ms each — ~20% latency/cost
# reduction with zero accuracy impact.
# audio_start_s forwarded from the Ears utterance to turn + alert events so the
# GUI override panel can seek the audio player to the exact escalation moment.
# ── Speaker identification — 3-tier priority ────────────────────────────────
# Tier 0 — stereo channel metadata (ground truth, used when available):
#   The Ears cell splits stereo recordings by channel and tags each chunk with
#   role='rep' (ch 0) or role='customer' (ch 1).  Brain reads this directly;
#   no regex or LLM is consulted.
# Tier 1 — keyword regex pre-pass for mono (_guess_mono_speaker, zero tokens):
#   Matches obvious lexical markers in the chunk text.  Customer markers:
#   "your agent/company", "loyal member", "she threatened", "I was just on
#   the phone".  Rep markers: "how can I help", "let me pull up", "one moment",
#   "you were speaking with".  Returns 'rep', 'customer', or None (ambiguous).
# Tier 2 — LLM judge speaker field (fallback for ambiguous mono chunks only):
#   System prompt tells the judge to identify the DOMINANT speaker and lean on
#   turn-alternation (last=rep → expect customer).  Prompt passes previous role
#   explicitly to reduce inference burden within the tight max_tokens=28 budget.
# In a live deployment the telephony/CTI layer supplies channel metadata and
# tiers 1 and 2 never run — speaker field is hard-coded from stream metadata.
# ─────────────────────────────────────────────────────────────────────────────
# ============ CELL 6: THE BRAIN ============
TURN_ANALYSIS_SCHEMA = {                                # the judge can ONLY answer in this shape
    "type": "object",
    "properties": {
        "speaker":    {"enum": ["rep", "customer"]},     # judge identifies who spoke
        "sentiment":  {"enum": [-2, -1, 0, 1, 2]},      # customer sentiment, clamped by grammar
        "violations": {"type": "array", "items": {"enum": list(POLICY)}},  # only real codes possible
        "reason":     {"type": "string", "maxLength": 35},   # 35 chars keeps decode tokens minimal
    },
    "required": ["speaker", "sentiment", "violations", "reason"],
}

_policy_lines = "\n".join(f"{code}: {definition} [{severity}]"            # compact policy digest for the prompt
                          for code, (severity, definition) in POLICY.items())
MODERATOR_SYSTEM_PROMPT = (                             # ~300 tokens, sent with every judge call
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST utterance, using context.\n"
    "/no_think\n"                                      # Qwen3: disable chain-of-thought — direct JSON only
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + COMPANY_POLICY_ALLOWANCES + "\n"
    "A venting customer is NOT a violation unless threats/abuse (C2). A rep REFUSING an improper "
    "request is NOT a violation. Routine identity verification (sending a verification text/link/code; asking for a name, email, phone, order number, or last-4 digits) is NOT C1 and NOT a violation — C1 applies ONLY to requesting or revealing a FULL SSN, FULL card number, CVV, or password. Report sentiment as the customer's sentiment right now (-2..2).\n"
    "Also set speaker: 'rep' if the utterance is from a call-center agent or scammer posing as authority (greeting callers, offering help, apologising on behalf of company, saying 'How can I help', 'I apologise', 'let me pull up'); 'customer' if the utterance is from the caller or victim (complaining about 'your agent/company', saying 'I was just on the phone', 'I'm so sorry', 'I'm a loyal member', expressing frustration/upset). On mono recordings both voices may appear in one chunk — identify the DOMINANT speaker. Speakers typically alternate: if context shows the last turn was rep, lean toward customer and vice versa.")

# Old scoreboard approach (REP_MARKER_PHRASES / CUSTOMER_MARKER_PHRASES) removed —
# replaced by _REP_RE / _CUSTOMER_RE regex pre-pass below.  Scoreboard accumulated
# phrase hits across turns; regex is per-chunk and faster.


# ── Mono speaker pre-classifier ───────────────────────────────────────────────
# LLM at max_tokens=28 with guided JSON + temperature=0 doesn't have enough
# headroom to reason about speaker identity AND produce the full schema — it
# defaults to "rep" every time.  A fast regex pre-pass handles the clear cases
# (one speaker's voice dominates the chunk); LLM fallback only for ambiguous.
_CUSTOMER_RE = re.compile(
    # ── Complaining about a specific agent ───────────────────────────────────
    r"your (?:agent|rep|representative|company|service|team|staff)|"
    r"(?:i'?m? a |i am a )(?:loyal|long.?time)\b|loyal member|"
    r"(?:she|he) (?:was |is )?(?:so |really |very )?(?:rude|condescending|mean|unprofessional|nasty)|"
    r"(?:she|he) threatened|threatened to (?:hang|disconnect|transfer|terminate)|"
    r"i was just on the phone|i didn'?t mean to|"
    r"i think (?:her|his) name was|"
    r"(?:you guys|you people)\b|"
    r"my (?:account|bill|order|subscription|service)\b|"
    # ── Customer asserting authority / being aggressive ───────────────────────
    r"you work for me\b|"                              # "you work for me, you're a service worker"
    r"i dare you\b|"                                   # "i dare you to disconnect"
    r"the customer is always|as the customer\b|"       # "the customer is always right", "as the customer"
    r"you have no rights\b|"                           # "you have no rights"
    r"(?:don'?t|do not) interrupt(?: me)?\b|"          # "do not interrupt me" (customer to rep)
    r"i don'?t care (?:what you|how i|if you)|"         # "I don't care what you have to do"
    # ── Customer identity markers ─────────────────────────────────────────────
    r"(?:platinum|gold|silver|diamond)\s+(?:card|member)|"  # "Platinum card holder"
    r"(?:card|account)\s+holder\b|"                   # "card holder", "account holder"
    r"i am an? (?:american|citizen|paying customer)\b|"  # "I am an American [card holder]"
    # ── Demanding escalation ──────────────────────────────────────────────────
    r"get me (?:over to |to )?(?:a |your |the )?(?:manager|supervisor)|"
    r"(?:i want|i need|i'd like) (?:to (?:speak|talk) (?:to |with )?)?(?:a |your |the )?(?:manager|supervisor)\b",
    re.IGNORECASE
)
_REP_RE = re.compile(
    r"how can i help|"
    r"let me (?:pull up|check|look into|review|bring up)|"
    r"(?:hold on|one moment|just a moment|one sec)\b|"
    r"(?:thank you|thanks) for (?:calling|holding|your patience)|"
    r"i(?:'m| am) (?:the |your |an? )?(?:escalations|customer service|support|manager|supervisor)|"
    r"you were speaking with|"
    r"i(?:'m| am) (?:pulling up|looking into|reviewing)|"
    r"i can (?:see|pull|check|verify)\b",
    re.IGNORECASE
)

def _guess_mono_speaker(text: str):
    """Regex pre-pass for mono recordings.  Returns 'rep', 'customer', or None (ambiguous)."""
    has_cust = bool(_CUSTOMER_RE.search(text))
    has_rep  = bool(_REP_RE.search(text))
    if has_cust and not has_rep:
        return "customer"
    if has_rep and not has_cust:
        return "rep"
    return None                         # both or neither — LLM decides
# ─────────────────────────────────────────────────────────────────────────────
def new_call_state():                                   # everything the Brain remembers about one call
    return {"context": deque(maxlen=3),                 # rolling window: last 3 utterances only (token diet)
            # "role_scores": defaultdict(int),          # removed — role determined by LLM judge
            "role_by_speaker": {},                      # speaker_tag -> last confirmed role (fast-path fallback)
            "violations_log": [],                       # [(turn_number, role, code)] for the rules + audit
            "sentiment_history": [],                    # customer sentiment trajectory (rule 3)
            "turn_number": 0,                           # running counter
            "escalated": False}                         # flag latch — first trigger wins

# guess_role() replaced by LLM judge speaker field — accurate on mono and stereo.
# def guess_role(state, speaker_tag, text):
#     ... marker-phrase scoreboard logic removed ...
#     return "rep" if speaker_tag == best_rep else "customer"

CALL_STATE = {}                                         # call_id -> state (the in-memory 'db' of live calls)

async def brain_worker(call_id: str, transcript_queue: asyncio.Queue, alert_queue: asyncio.Queue):
    """STEP 2: await queue.get() -> judge -> rules -> (maybe) alert. One worker per call."""
    state = CALL_STATE.setdefault(call_id, new_call_state())
    while True:
        item = await transcript_queue.get()             # sleeps here until THE EARS delivers (or sentinel)
        if item is None:                                # sentinel = this call's audio is finished
            break
        dequeued_at = time.perf_counter()               # when the Brain actually picked this turn up
        state["turn_number"] += 1
        turn_number = state["turn_number"]
        try:
            import langfuse_config as _lfc
            _lfc.set_current_call_id(call_id)
        except Exception:
            pass
        _last_role = state.setdefault("role_by_speaker", {}).get(item["speaker"], "rep")  # setdefault guards against stale state after hot-reload
        hints = keyword_prescreen(item["text"])         # 0-token candidate codes for the judge to verify
        # ==== PERFORMANCE OVERHAUL 2: ack fast-path ====
        # Micro-utterances ("Okay.", "Yes.", "Nope. No.") carry zero policy signal, yet each
        # was costing a full judge round-trip (~4s on this contended GPU) and ~350 tokens.
        # They are a big share of real calls, so skipping them is a major latency+token win.
        # The regex pre-screener OVERRIDES the shortcut: a short-but-dangerous utterance
        # (e.g. "CVV?") still goes to the judge.
        if len(item["text"].strip()) <= 12 and not hints:
            analysis = {"sentiment": 0, "violations": [], "reason": "", "speaker": _last_role}  # carry last known role
        else:
            hint_note = f"\nRegex hints to verify (may be false alarms): {hints}" if hints else ""
            context = "\n".join(state["context"]) or "(call start)"      # last 3 utterances max
            # ==== PERFORMANCE OVERHAUL 3: leaner decode ====
            # max_tokens=48: covers the max plausible verdict (~35 tokens for two
            # violations + full 35-char reason).  Lossless — vLLM guided JSON stops
            # at EOS when the schema closes (~18-22 tokens for typical verdicts),
            # so latency is unchanged vs 28; we just eliminate truncation on edge cases.
            # stereo: tell LLM who is already speaking so it has accurate context
            # mono:   ask LLM to identify speaker from content cues
            _channel_role = item.get("role")      # None for mono, 'rep'/'customer' for stereo
            _mono_hint    = None
            if _channel_role is not None:
                # stereo: channel index is ground truth, no guessing needed
                _role_label = f"{'Rep' if _channel_role == 'rep' else 'Customer'} [stereo — channel-assigned]"
            else:
                # mono tier 1: fast regex — reliable for clear cases
                _mono_hint = _guess_mono_speaker(item["text"])
                _other     = "customer" if _last_role == "rep" else "rep"
                if _mono_hint is not None:
                    _role_label = (
                        f"{'Customer' if _mono_hint == 'customer' else 'Rep'} "
                        f"[mono — keyword match; confirm from content]"
                    )
                else:
                    # mono tier 2: LLM fallback with alternation hint
                    _role_label = (
                        f"Unknown speaker [mono — previous turn was '{_last_role}', "
                        f"likely '{_other}' now; identify from content cues]"
                    )
            analysis = await generate_json("turn_moderation", MODERATOR_SYSTEM_PROMPT,
                                           f'Context:\n{context}\n\nLATEST utterance ({_role_label}): '
                                           f'"{item["text"]}"{hint_note}',
                                           TURN_ANALYSIS_SCHEMA, max_tokens=48)  # max plausible verdict ~35 tokens; 48 covers all cases losslessly
        judged_at = time.perf_counter()
        _channel_role_final = item.get("role")
        if _channel_role_final is not None:
            role = _channel_role_final          # stereo: ground truth
        elif _mono_hint is not None:
            role = _mono_hint                   # mono tier 1: keyword regex (reliable)
        else:
            role = analysis.get("speaker", _last_role)  # mono tier 2: LLM fallback
        state["role_by_speaker"][item["speaker"]] = role  # cache for fast-path
        state["context"].append(f"{role}: {item['text']}")               # update rolling window AFTER judging
        violations = [c for c in analysis.get("violations", []) if c in POLICY]
        sentiment = analysis.get("sentiment", 0)
        state["violations_log"] += [(turn_number, role, c) for c in violations]
        if role == "customer":
            state["sentiment_history"].append(sentiment)
        queue_wait_ms = (dequeued_at - item["transcribed_at"]) * 1000   # time spent waiting in the FIFO
        judge_ms      = (judged_at - dequeued_at) * 1000                # pure LLM verdict time
        total_ms      = item.get("asr_ms", 0) + queue_wait_ms + judge_ms  # spoken-audio -> verdict logged
        violation_note = f"   <- {violations}: {analysis.get('reason', '')}" if violations else ""
        print(f"[{call_id} t{turn_number:02d}] {role:>8} ({sentiment:+d}): "
              f"{item['text']}{violation_note}\n"
              f"        timing: asr {item.get('asr_ms', 0):.0f}ms | queue {queue_wait_ms:.0f}ms | "
              f"judge {judge_ms:.0f}ms | audio->verdict {total_ms:.0f}ms", flush=True)
        await emit_event({"type": "turn", "call_id": call_id, "turn_number": turn_number,
                          "role": role, "text": item["text"], "sentiment": sentiment,
                          "violations": violations, "reason": analysis.get("reason", ""),
                          "asr_ms": round(item.get("asr_ms", 0)), "queue_ms": round(queue_wait_ms),
                          "judge_ms": round(judge_ms),
                          "audio_start_s": item.get("audio_start_s", 0)})  # GUI audio seek
        audit_db.execute("INSERT INTO turns VALUES (?,?,?,?,?,?,?)",     # in-memory audit trail, every turn
                         (call_id, turn_number, role, item["text"],
                          sentiment, ",".join(violations), analysis.get("reason", "")))
        if not state["escalated"]:                      # the three deterministic rules, in priority order
            severities = [SEVERITY_BY_CODE[c] for _, _, c in state["violations_log"]]
            recent = state["sentiment_history"][-2:]
            rule = None
            if "critical" in severities:
                rule = f"rule 1: critical violation {violations}"
            elif severities.count("high") >= 2:
                rule = "rule 2: two high-severity violations"
            elif len(recent) == 2 and all(s <= -2 for s in recent):
                rule = "rule 3: sustained very negative customer sentiment"
            if rule:
                state["escalated"] = True               # latch — one alarm per call
                await alert_queue.put({                 # ZERO-COPY handshake 2->3
                    "call_id": call_id, "turn_number": turn_number, "rule": rule,
                    "detail": f'{role}: "{item["text"]}"',
                    "audio_start_s": item.get("audio_start_s", 0),       # GUI: seek to escalation moment
                    "transcribed_at": item["transcribed_at"],            # end-to-end clock start
                    "dequeued_at": dequeued_at,                           # separates queue wait from judge time
                    "flagged_at": judged_at})

print("THE BRAIN ready (1 worker per call, last-3-utterance context, deterministic rules)")

## CELL 7 — STEP 3: THE ALARM (instant flag + in-memory audit DB)

Replaces the document's PyQt6 red-flash window with what Jupyter can do **instantly**: a
flushed, unmissable alert line the moment the Brain pushes to the alert queue, plus a row in
a **sqlite `:memory:`** database (the in-memory DB) so every flag and every judged turn is
queryable afterwards. The printed latency is the headline metric: *transcript ready → flag
raised*, in milliseconds.

In [ ]:
# ── Why sqlite :memory:, flush=True print, async queue handshake ───────────────
# sqlite :memory:: zero disk I/O in the hot path; the full audit trail is
# queryable with standard SQL after the run (CELL 9/10).  Swap to a file-path
# DB for cross-run persistence.
# flush=True: bypasses Python's stdout buffer so the alert line appears in the
# notebook cell output THE INSTANT it is printed — the visual signal IS the
# latency measurement for a supervisor watching live.
# Queue handshake: the Brain pushes to alert_queue and immediately continues
# to the next turn — it never waits for the Alarm to print or write to DB,
# keeping judge latency clean of I/O cost.
# ============ CELL 7: THE ALARM ============
audit_db = sqlite3.connect(":memory:")                  # in-memory DB: zero I/O latency, queryable with SQL
audit_db.execute("CREATE TABLE turns  (call_id TEXT, turn_number INT, role TEXT, text TEXT, "
                 "sentiment INT, violations TEXT, reason TEXT)")        # every judged utterance
audit_db.execute("CREATE TABLE alerts (call_id TEXT, turn_number INT, rule TEXT, detail TEXT, "
                 "flag_latency_ms REAL)")                               # every raised flag

FLAG_LATENCIES_MS = []                                  # collected for the final stats cell

async def alarm_consumer(alert_queue: asyncio.Queue):
    """STEP 3: await alert -> print INSTANTLY (flush=True) + record to the audit DB."""
    while True:
        alert = await alert_queue.get()                 # sleeps until the Brain raises a flag (or sentinel)
        if alert is None:                               # sentinel = pipeline shutting down
            break
        flag_latency_ms  = (alert["flagged_at"] - alert["transcribed_at"]) * 1000  # end-to-end
        queue_wait_ms    = (alert.get("dequeued_at", alert["transcribed_at"]) - alert["transcribed_at"]) * 1000
        judge_ms         = (alert["flagged_at"] - alert.get("dequeued_at", alert["transcribed_at"])) * 1000
        FLAG_LATENCIES_MS.append(judge_ms)              # the honest real-time number: pure judge latency
        audit_db.execute("INSERT INTO alerts VALUES (?,?,?,?,?)",
                         (alert["call_id"], alert["turn_number"], alert["rule"],
                          alert["detail"], flag_latency_ms))
        print("\n" + "!" * 78, flush=True)              # flush=True -> appears the instant it happens
        print(f"!! ESCALATE  {alert['call_id']}  turn {alert['turn_number']}  ({alert['rule']})", flush=True)
        print(f"!! {alert['detail']}", flush=True)
        print(f"!! judge latency: {judge_ms:.0f} ms  (+{queue_wait_ms:.0f} ms queue wait in "
              f"flat-out benchmark mode; end-to-end {flag_latency_ms:.0f} ms)", flush=True)
        # NOTE: queue wait exists because REALTIME_PACING=False floods all chunks at once.
        # With live pacing (1 chunk / 5s) the queue stays empty whenever judge < 5s.
        print("!" * 78 + "\n", flush=True)
        await emit_event({"type": "alert", "call_id": alert["call_id"],
                          "turn_number": alert["turn_number"], "rule": alert["rule"],
                          "detail": alert["detail"], "judge_ms": round(judge_ms),
                          "audio_start_s": alert.get("audio_start_s", 0)})  # GUI override seek

print("THE ALARM ready (instant flush + sqlite :memory: audit)")

## CELL 8 — Run the assembly line

Wires everything: one transcript queue + one Ears producer + one Brain worker **per call**,
one shared alert queue + one Alarm consumer, all under a single `asyncio.gather`. With
`REALTIME_PACING=False` this is a flat-out benchmark; set it `True` in CELL 5 to watch flags
fire against a simulated live stream.

In [ ]:
# ── Why asyncio.gather + one queue per call + shared alert queue ────────────────
# asyncio.gather runs all producer+worker pairs concurrently on one thread — no
# multiprocessing overhead, no shared-memory synchronization; the event loop
# multiplexes I/O waits (queue.get, LLM responses) across all lanes.
# One transcript_queue per call: FIFO order within a call is preserved while
# the shared GPU is utilized across calls — no call blocks another.
# Shared alert_queue: a single Alarm consumer serializes all escalations from
# all calls so output is readable and only one DB connection is needed.
# Sentinel None signals EOF cleanly — any get() returning None means 'done'.
# ============ CELL 8: orchestration ============
# ── Clear the dashboard for this fresh run ──────────────────────────────────
# POSTs to gui_server /clear which wipes EVENT_LOG, rewrites the event file,
# and pushes a reset signal to every open browser tab so old recordings
# disappear instantly without needing to refresh or restart gui_server.
import urllib.request, urllib.error
try:
    urllib.request.urlopen(
        urllib.request.Request('http://localhost:7860/clear', method='POST')
    )
    print('[dashboard] cleared — starting fresh run')
except urllib.error.URLError:
    pass  # gui_server not running — non-fatal

# ── Start a fresh Langfuse session for this pipeline run ────────────────────
# new_session() regenerates _session_id so each run appears as a separate
# Session in the Langfuse UI (module import only runs once per kernel start).
try:
    langfuse_config.new_session()
except Exception:
    pass  # Langfuse cell not run — non-fatal

REALTIME_SINGLE_CALL = True    # ====== THE MODE TOGGLE ======
                               # True  -> real-time single call, paced at 1x
                               # False -> bulk: flat-out, full-accuracy ASR

# ── Recording selection ───────────────────────────────────────────────────────
# Single mode:
#   SELECTED_CALL_IDS = "scam_call_2_gemma3n_ready"  -> process that recording
#   SELECTED_CALL_IDS = ""                            -> pick one at random from AUDIO_DIRS
#
# Bulk mode:
#   SELECTED_CALL_IDS = "call_a,scam_call_1_gemma3n_ready"  -> process those two
#   SELECTED_CALL_IDS = ""                                   -> process ALL in AUDIO_DIRS
#
# Error: if a name is given but not found, the cell prints available IDs and stops.
SELECTED_CALL_IDS = ""          # stem name(s), no extension; comma-separated for bulk
# ─────────────────────────────────────────────────────────────────────────────

import random

if not audio_files:
    raise RuntimeError(
        "No audio files found.\nCheck AUDIO_DIRS in CELL 4 and ensure recordings exist."
    )

_stem_map = {p.stem: p for p in audio_files}           # stem -> Path for fast lookup
print("available call ids:", list(_stem_map.keys()))

_requested = [s.strip() for s in SELECTED_CALL_IDS.split(",") if s.strip()]

def _resolve(names: list, available: dict) -> list:
    if not names:
        return list(available.values())
    resolved, missing = [], []
    for n in names:
        (resolved if n in available else missing).append(n)
    if missing:
        print(f"ERROR: recording(s) not found in any AUDIO_DIR: {missing}")
        print(f"Available ids: {list(available.keys())}")
        raise ValueError(f"Recording(s) not found: {missing}")
    return [available[n] for n in names]

REALTIME_PACING = REALTIME_SINGLE_CALL
print(f"[mode] REALTIME_PACING={REALTIME_PACING} | USE_GPU_STT={USE_GPU_STT} — "
      f"{'GPU STT live-stream' if (REALTIME_PACING and USE_GPU_STT) else 'CPU faster-whisper file mode'}")

if REALTIME_SINGLE_CALL:
    if _requested:
        selected_recordings = _resolve(_requested[:1], _stem_map)
        print(f"REAL-TIME single call: {selected_recordings[0].stem} (paced at 1x)")
    else:
        selected_recordings = [random.choice(list(_stem_map.values()))]
        print(f"REAL-TIME single call: {selected_recordings[0].stem} (random pick, paced at 1x)")
    if WHISPER_MODEL_SIZE not in ("large-v3-turbo", "distil-large-v3", "small"):
        print(f"note: {WHISPER_MODEL_SIZE} may lag 5s pacing — large-v3-turbo recommended")
else:
    if _requested:
        selected_recordings = _resolve(_requested, _stem_map)
        print(f"BULK moderation: {len(selected_recordings)} specified recording(s): "
              f"{[p.stem for p in selected_recordings]}")
    else:
        selected_recordings = list(_stem_map.values())
        print(f"BULK moderation: all {len(selected_recordings)} recordings across AUDIO_DIRS, "
              f"flat-out (full-file ASR per channel)")

CALL_STATE.clear()                                      # reset per-call state between runs
try:
    import langfuse_config as _lfc; _lfc.clear_traces()
except Exception:
    pass
alert_queue = asyncio.Queue()                           # shared Brain -> Alarm channel
alarm_task  = asyncio.create_task(alarm_consumer(alert_queue))

pipeline_started = time.perf_counter()
per_call_tasks = []
for audio_path in selected_recordings:
    call_id = audio_path.stem
    transcript_queue = asyncio.Queue()
    per_call_tasks.append(ears_producer(call_id, audio_path, transcript_queue))
    per_call_tasks.append(brain_worker(call_id, transcript_queue, alert_queue))

await asyncio.gather(*per_call_tasks)
await alert_queue.put(None)
await alarm_task
pipeline_seconds = time.perf_counter() - pipeline_started

total_turns = audit_db.execute("SELECT COUNT(*) FROM turns").fetchone()[0]
print(f"assembly line done: {len(selected_recordings)} call(s), {total_turns} judged utterances "
      f"in {pipeline_seconds:.1f}s ({total_turns/max(pipeline_seconds,0.001):.1f} utterances/s)")


## CELL 9 — Results: flags, latency, tokens

Three readouts, all from in-memory data: the **alerts** (what was flagged, which rule, how
fast), **latency** percentiles per stage (judge latency *is* the flag latency), and exact
**token usage** per stage with a combined total.

In [ ]:
# ── Why a hand-rolled percentile helper instead of numpy ────────────────────────
# numpy is already loaded as a transitive dependency, but importing it here for
# a single percentile would add an unnecessary coupling to this results cell.
# The 4-line sort+index helper is exact, zero-dependency, and readable to anyone
# auditing the latency reporting — which is the whole point of this cell.
# ============ CELL 9: results ============
print("RAISED FLAGS")                                   # straight from the in-memory audit DB
for row in audit_db.execute("SELECT call_id, turn_number, rule, flag_latency_ms FROM alerts"):
    print(f"  {row[0]:<28} turn {row[1]:<3} {row[2]:<48} {row[3]:>7.0f} ms")
if not FLAG_LATENCIES_MS:
    print("  (no escalations triggered on these recordings)")

def percentile(values, fraction):                       # tiny helper — avoids a numpy dependency here
    ordered = sorted(values)
    return ordered[min(int(len(ordered) * fraction), len(ordered) - 1)]

print("\nLATENCY (ms)")                                 # judge latency == transcript->flag latency per turn
for stage, samples in STAGE_LATENCY_MS.items():
    print(f"  {stage:<22} n={len(samples):<5} avg={sum(samples)/len(samples):>7.0f}  "
          f"p95={percentile(samples, 0.95):>7.0f}  max={max(samples):>7.0f}")
if FLAG_LATENCIES_MS:
    print(f"  {'transcript->flag':<22} n={len(FLAG_LATENCIES_MS):<5} "
          f"avg={sum(FLAG_LATENCIES_MS)/len(FLAG_LATENCIES_MS):>7.0f}  "
          f"max={max(FLAG_LATENCIES_MS):>7.0f}")

# What each column means:
#   calls      = number of LLM judge invocations at this pipeline stage
#   prompt     = tokens sent IN per call (system prompt ~300 + 3-turn context + current turn ~50-100)
#   completion = tokens returned per call — the JSON verdict is tiny:
#                {"sentiment":-1,"violations":["C2"],"reason":"..."} ~ 20-30 tokens
#   total      = prompt + completion; multiply by your provider's $/token for cost estimate
# Example: 1 call to the judge for a single turn costs ~350 prompt + ~25 completion = ~375 tokens.
print("\nTOKENS BY STAGE  (1 call = 1 LLM judge invocation for one conversation turn)")
grand = {"calls": 0, "prompt": 0, "completion": 0}
for stage, usage in sorted(STAGE_TOKEN_USAGE.items()):
    total = usage["prompt"] + usage["completion"]
    print(f"  {stage:<22} calls={usage['calls']:<5} prompt={usage['prompt']:>8,} "
          f"completion={usage['completion']:>8,} total={total:>9,}")
    for field in grand:
        grand[field] += usage[field]
print(f"  {'COMBINED':<22} calls={grand['calls']:<5} prompt={grand['prompt']:>8,} "
      f"completion={grand['completion']:>8,} total={grand['prompt']+grand['completion']:>9,}")

print("\nPER-CALL SUMMARY")                             # SQL over the in-memory audit trail
for row in audit_db.execute(
    "SELECT call_id, COUNT(*), SUM(violations != ''), MIN(sentiment) FROM turns GROUP BY call_id"):
    print(f"  {row[0]:<28} turns={row[1]:<4} turns_with_violations={row[2] or 0:<3} "
          f"worst_sentiment={row[3]}")
# ── LANGFUSE SESSION SUMMARY ──────────────────────────────────────────────────
try:
    import langfuse_config as _lf_cfg
    _traces_all = []
    for _cid in _lf_cfg.get_all_call_ids():
        _traces_all.extend(_lf_cfg.get_call_traces(_cid))

    if _traces_all:
        _sid   = _lf_cfg._session_id
        _mode  = _lf_cfg._mode
        _url   = _lf_cfg._url if hasattr(_lf_cfg, '_url') else ''

        print(f"\nLANGFUSE SESSION")
        print(f"  session_id : {_sid}")
        print(f"  mode       : {_mode}" + (f"  ({_url})" if _url else ""))
        print(f"  generations: {len(_traces_all)}")

        # per-stage aggregation
        _by_stage: dict = {}
        for _r in _traces_all:
            _s = _r.get("stage", "?")
            _u = _r.get("usage", {})
            _e = _r.get("elapsed_ms", 0)
            if _s not in _by_stage:
                _by_stage[_s] = {"calls": 0, "input": 0, "output": 0, "elapsed_ms": []}
            _by_stage[_s]["calls"]      += 1
            _by_stage[_s]["input"]      += _u.get("input", 0)
            _by_stage[_s]["output"]     += _u.get("output", 0)
            _by_stage[_s]["elapsed_ms"].append(_e)

        print(f"\n  {'stage':<22} {'calls':>5}  {'in_tok':>7}  {'out_tok':>7}  {'total_tok':>9}  {'avg_ms':>7}")
        print(f"  {'-'*22}  {'-'*5}  {'-'*7}  {'-'*7}  {'-'*9}  {'-'*7}")
        _gt_in = _gt_out = _gt_calls = 0
        for _s, _d in sorted(_by_stage.items()):
            _tot  = _d["input"] + _d["output"]
            _avg  = sum(_d["elapsed_ms"]) / len(_d["elapsed_ms"]) if _d["elapsed_ms"] else 0
            _gt_in    += _d["input"]
            _gt_out   += _d["output"]
            _gt_calls += _d["calls"]
            print(f"  {_s:<22} {_d['calls']:>5}  {_d['input']:>7,}  {_d['output']:>7,}  {_tot:>9,}  {_avg:>7.0f}")
        _gt_tot = _gt_in + _gt_out
        print(f"  {'TOTAL':<22} {_gt_calls:>5}  {_gt_in:>7,}  {_gt_out:>7,}  {_gt_tot:>9,}")

        # per-call breakdown
        _by_call: dict = {}
        for _r in _traces_all:
            _cid2 = _r.get("call_id", "?")
            _u    = _r.get("usage", {})
            _e    = _r.get("elapsed_ms", 0)
            if _cid2 not in _by_call:
                _by_call[_cid2] = {"gen": 0, "tokens": 0, "elapsed_ms": []}
            _by_call[_cid2]["gen"]    += 1
            _by_call[_cid2]["tokens"] += _u.get("total", 0)
            _by_call[_cid2]["elapsed_ms"].append(_e)

        print(f"\n  {'call_id':<28} {'gen':>4}  {'tokens':>7}  {'avg_judge_ms':>12}")
        for _cid2, _d in sorted(_by_call.items()):
            _avg = sum(_d["elapsed_ms"]) / len(_d["elapsed_ms"]) if _d["elapsed_ms"] else 0
            print(f"  {_cid2:<28} {_d['gen']:>4}  {_d['tokens']:>7,}  {_avg:>12.0f}")

        if _mode != "local":
            print(f"\n  View in Langfuse: {_url}/sessions/{_sid}")
    else:
        print("\nLANGFUSE SESSION  — no generations recorded (langfuse cell may not have run)")
except Exception as _lf_err:
    print(f"\nLANGFUSE SESSION  — unavailable ({_lf_err})")


## Design notes

- **Why `asyncio.Queue` and not LangChain/LangGraph:** the pipeline is a fixed 3-stage line —
  the architecture document itself specifies in-process queues. A graph framework would add
  abstraction layers (and milliseconds) between transcript and flag for zero functional gain.
- **Why one queue/worker per call:** FIFO order within a call is required (rule 3 reads
  *consecutive* sentiment), while calls stay parallel and vLLM batches across them. The
  shared `Semaphore(16)` is the global concurrency gate.
- **Where the speed comes from:** silence chunks are dropped before transcription (0 cost),
  the prompt carries only the last 3 utterances, guided JSON kills retries, and the alarm
  consumer prints with `flush=True` the instant the Brain pushes — flag latency is one judge
  call (~hundreds of ms on a 4B model), measured and reported per flag.
- **To go truly live:** replace `ears_producer`'s file loop with a WebSocket/WebRTC receiver
  pushing 5s chunks — every line downstream of `transcript_queue.put(...)` stays identical.
- **Real-time rehearsal:** set `REALTIME_PACING = True` in CELL 5 to pace chunks at 1×.

## CELL 11 — Replay the run onto the dashboard

If the GUI was restarted (or down) during a run, the dashboard missed those events.
This cell re-emits **every judged turn and escalation from the audit DB**, repopulating
the dashboard exactly as it would have looked live. Run CELL 3b first if the kernel
still has the old emitter loaded.

In [ ]:
# ============ CELL 11: replay audit DB -> dashboard ============
replayed_turns = 0
for call_id_, turn_no_, role_, text_, sentiment_, violations_, reason_ in audit_db.execute(
        "SELECT call_id, turn_number, role, text, sentiment, violations, reason "
        "FROM turns ORDER BY call_id, turn_number"):
    await emit_event({"type": "turn", "call_id": call_id_, "turn_number": turn_no_,
                      "role": role_, "text": text_, "sentiment": sentiment_,
                      "violations": violations_.split(",") if violations_ else [],
                      "reason": reason_, "asr_ms": 0, "queue_ms": 0, "judge_ms": 0})
    replayed_turns += 1
replayed_alerts = 0
for call_id_, turn_no_, rule_, detail_, latency_ in audit_db.execute(
        "SELECT call_id, turn_number, rule, detail, flag_latency_ms FROM alerts"):
    await emit_event({"type": "alert", "call_id": call_id_, "turn_number": turn_no_,
                      "rule": rule_, "detail": detail_, "judge_ms": round(latency_ or 0)})
    replayed_alerts += 1
print(f"replayed {replayed_turns} turns + {replayed_alerts} alerts to the dashboard "
      f"(GUI_AVAILABLE={GUI_AVAILABLE})")